In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

Data preparation

In [2]:
URL = ("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
       "main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [3]:
df = pd.read_csv(URL)
print("Missing values:\n", df.isnull().sum(), "\n")

Missing values:
 lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64 



In [5]:
target = "converted"
numerical = df.drop(columns=target).select_dtypes(include="number").columns.tolist()
categorical = df.drop(columns=target).select_dtypes(exclude="number").columns.tolist()

print(numerical)
print(categorical)

['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
['lead_source', 'industry', 'employment_status', 'location']


In [7]:
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)
 
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
 
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

print(y_train.mean(), y_val.mean())
print(y_val.mean(), y_test.mean())
print(y_test.mean(), y_train.mean())
print(df_train.shape, df_val.shape, df_test.shape)

0.5763333333333334 0.586
0.586 0.568
0.568 0.5763333333333334
(3000, 9) (1000, 9) (1000, 9)


ROC AUC feature importance 

In [8]:
print("Q1")
for col in ["lead_score", "number_of_courses_viewed", "interaction_count", "annual_income"]:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    print(f"  {col}: {auc:.3f}")
 

Q1
  lead_score: 0.788
  number_of_courses_viewed: 0.723
  interaction_count: 0.765
  annual_income: 0.608


In [9]:
features = categorical + numerical
 
def train(df_tr, y_tr, C=1.0):
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(df_tr[features].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
    model.fit(X, y_tr)
    return dv, model
 
def predict(df_v, dv, model):
    X = dv.transform(df_v[features].to_dict(orient="records"))
    return model.predict_proba(X)[:, 1]

Model AUC 

In [10]:
dv, model = train(df_train, y_train)
y_pred = predict(df_val, dv, model)
print(f"\nQ2: AUC validation = {roc_auc_score(y_val, y_pred):.3f}")


Q2: AUC validation = 0.732


In [11]:
thresholds = np.arange(0.0, 1.01, 0.01)
rows = []
for t in thresholds:
    pred = y_pred >= t
    tp = ((pred == 1) & (y_val == 1)).sum()
    fp = ((pred == 1) & (y_val == 0)).sum()
    fn = ((pred == 0) & (y_val == 1)).sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    rows.append((t, p, r, f1))
 
scores = pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])
 
valid = scores[(scores.precision > 0) | (scores.recall > 0)].copy()
valid["diff"] = (valid.precision - valid.recall).abs()
q3 = valid.loc[valid["diff"].idxmin(), "threshold"]
print(f"Q3: precision and recall intersect at threshold = {q3:.2f}")

Q3: precision and recall intersect at threshold = 0.63


Precision and recall 

In [12]:
q4 = scores.loc[scores.f1.idxmax(), "threshold"]
print(f"Q4: max F1 at threshold = {q4:.2f}")

Q4: max F1 at threshold = 0.41


F1 score

In [13]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
aucs = []
for tr_idx, va_idx in kfold.split(df_full_train):
    d_tr = df_full_train.iloc[tr_idx]
    d_va = df_full_train.iloc[va_idx]
    dv_k, m_k = train(d_tr, d_tr[target].values)
    aucs.append(roc_auc_score(d_va[target].values, predict(d_va, dv_k, m_k)))
print(f"Q5: std of AUC across folds = {np.std(aucs):.3f}")

Q5: std of AUC across folds = 0.007


5-Fold CV standard deviation

In [14]:
print("Q6")
results = []
for C in [0.000001, 0.001, 1]:
    aucs = []
    for tr_idx, va_idx in kfold.split(df_full_train):
        d_tr = df_full_train.iloc[tr_idx]
        d_va = df_full_train.iloc[va_idx]
        dv_k, m_k = train(d_tr, d_tr[target].values, C=C)
        aucs.append(roc_auc_score(d_va[target].values, predict(d_va, dv_k, m_k)))
    mean, std = round(np.mean(aucs), 3), round(np.std(aucs), 3)
    results.append((C, mean, std))
    print(f"  C={C}: mean={mean:.3f}, std={std:.3f}")

Q6
  C=1e-06: mean=0.617, std=0.019
  C=0.001: mean=0.749, std=0.010
  C=1: mean=0.732, std=0.007


Best C 

In [15]:
best = sorted(results, key=lambda r: (-r[1], r[2], r[0]))[0]
print(f"  Best C = {best[0]}")

  Best C = 0.001
